# Sphota GPU run 4: confirmation on unused data, better adapters, and > 256 cells (D-055, D-056)

Run 3 passed the decision gate narrowly at 256 cells, on a test set three runs had used. Run 4 does three things in one session, embedding once:

* **A. Confirmation (the gate test):** the two best run-3 configurations on **NTREX-128**, news sentences no run has used, with three fresh replicates.
* **B. Better shipped adapters (v1.3):** the current standard's adapters retrained with 10× more pairs plus short commands in 12 languages.
* **C. Improvement (exploratory):** 120k / 240k / 400k adapter pairs × two sizes × 256 / 512 / 1024 cells.

**Before you run** (right-hand panel → *Session options*): **Accelerator** GPU T4 ×2; **Internet** on. Then *Save Version → Save & Run All*.
**Time:** about 7–8 hours. A 10-minute smoke test runs first and stops the notebook if anything is broken. Parts are skipped automatically if time runs short; results are always saved.
**Cost:** ₹0 (free weekly GPU quota). **Download:** `sphota-run4.zip` from the Output tab.

In [ ]:
# 1. Get the code (public repository: no token needed, and none should be added)
import os, subprocess, time
os.environ["SPHOTA_DEADLINE"] = str(time.time() + 10.5 * 3600)  # Kaggle stops at 12 h: leave time to save
os.environ["SPHOTA_HF_CACHE"] = "/tmp/sphota-hf"    # models and embedding caches stay out of /kaggle/working
os.environ["SPHOTA_DATA_DIR"] = "/tmp/sphota-data"  # (its 20 GB output limit and the save step)
os.environ["PYTHONUNBUFFERED"] = "1"
if not os.path.exists("/kaggle/working/sphota"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Thiruvikramanpravartak/sphota.git", "/kaggle/working/sphota"], check=True)
os.chdir("/kaggle/working/sphota")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install (Kaggle already has CUDA torch; this adds the rest). Never `import sphota` in this notebook:
#    the kernel started before this install, so every use goes through `!python` (D-054).
!pip -q install -e ".[build,encode]"
import torch
os.environ["SPHOTA_THREADS"] = str(os.cpu_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on a GPU accelerator")
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# 3. Smoke test (about 10 min): every code path of parts A-C at tiny scale, every model, plus the
#    summary and the zip. Stops the notebook if anything fails.
import glob, json
!python -m sphota.build.data --scale tiny 2>&1 | tail -1
!python -m sphota.build.gate v3sweep --scale tiny --grid smoke4 --anchor BAAI/bge-m3 2>&1 | tee /kaggle/working/smoke.log | grep -E "^\[|PRIMARY|Error|error" | tail -6
!python -m sphota.build.adapters_v11 --scale tiny --set v1.3 --models sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 2>&1 | tee -a /kaggle/working/smoke.log | grep -E "L1 en|Error" | tail -2
f = "bench/results/v4-smoke-tiny-bge-m3-smoke4.json"
ok = os.path.exists(f)
if ok:
    r = json.load(open(f))
    ok = (len(r["rows"]) == 4 and r.get("primary_results") and all("evals" in x for x in r["rows"])
          and "cross_model_extra" in r["rows"][0] and os.path.exists("src/sphota/data/adapters/v1.3/pm-minilm-l12.npz"))
summary = subprocess.run(["python", "-m", "sphota.build.gate", "v3summary", f], capture_output=True, text=True)
ok = ok and summary.returncode == 0 and "PRIMARY" in summary.stdout
!rm -rf src/sphota/data/adapters/v1.3 bench/results/adapters-v1.3-tiny.json bench/results/v4-smoke-tiny-*.json
if not ok:
    raise RuntimeError("Smoke test FAILED - see /kaggle/working/smoke.log; nothing expensive was run. Send the log.")
print("smoke test OK")

In [ ]:
# 4. Data (medium scale, about 15 min)
!python -m sphota.build.data --scale medium 2>&1 | tail -1

In [ ]:
# 5. PART A - confirmation on NTREX-128 (D-055): C2 and C1, three fresh replicates. About 2.5 h
#    (includes embedding the shared 400k training pool once for every model).
!python -m sphota.build.gate v3sweep --scale medium --grid confirm4 --anchor BAAI/bge-m3 2>&1 | tee /kaggle/working/run4-A.log | grep -vE "it/s|sent/s|Batches|Warning|warn"

In [ ]:
# 6. PART B - v1.3 adapters for the frozen standard (D-056): 7 models, 400k pairs + commands. About 45 min.
!python -m sphota.build.adapters_v11 --scale medium --set v1.3 --models sentence-transformers/all-MiniLM-L6-v2 BAAI/bge-small-en-v1.5 thenlper/gte-small sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 sentence-transformers/LaBSE intfloat/multilingual-e5-base BAAI/bge-base-en-v1.5 2>&1 | tee /kaggle/working/run4-B.log | grep -vE "it/s|sent/s|Batches|Warning|warn"

In [ ]:
# 7. PART C - improvement sweep (exploratory): more adapter data and capacity, 256 / 512 / 1024 cells.
#    About 3.5 h; blocks past the 10.5 h deadline are skipped and recorded.
!python -m sphota.build.gate v3sweep --scale medium --grid improve4 --anchor BAAI/bge-m3 2>&1 | tee /kaggle/working/run4-C.log | grep -vE "it/s|sent/s|Batches|Warning|warn"

In [ ]:
# 8. Save everything to download (Output tab -> sphota-run4.zip): results, v1.3 adapters, logs. Runs before any summary.
!cd /kaggle/working && zip -qr sphota-run4.zip sphota/bench/results/v4-*-medium-*.json sphota/bench/results/adapters-v1.3-medium.json sphota/src/sphota/data/adapters/v1.3 run4-*.log smoke.log 2>/dev/null; ls -la /kaggle/working/sphota-run4.zip
print("Download sphota-run4.zip from the Output tab and upload it to the chat.")

In [ ]:
# 9. Results summary, in its own process (the zip above is already written)
!python -m sphota.build.gate v3summary --scale medium 2>&1 | grep -E "^sweep|PRIMARY|exploratory|INCOMPLETE|skipped|^=|json$"
!grep -E "L1 en" /kaggle/working/run4-B.log | cut -c1-200

## Reading the result
- **Part A: a configuration CONFIRMED (all three replicates pass on NTREX)** → the 256-cell code holds on unused data; the next step is an RFC for space `s2` (D-017).
- **Not confirmed** → run 3's pass is reported as not replicated; positioning and the v1 standard stay as they are.
- **Part B** is decided on CPU after download (D-056). **Part C** shows what more adapter data buys at 512 and 1024 cells; leads need their own confirmation.
Pre-registration: `DECISIONS.md` D-055, D-056.